# Lions vs Tigers Image Classification

This notebook mirrors the Cats vs Dogs (mini-Xception) workflow, adapted for a **Lion vs Tiger** binary image classifier.

## Getting the dataset

Use a Kaggle Lion/Tiger dataset, e.g. `akrsnv/lions-and-tigers` (400 images, 256x256, 2 classes). You'll need a Kaggle API token (`kaggle.json`) placed in `~/.kaggle/`.

```shell
pip install kaggle
kaggle datasets download -d akrsnv/lions-and-tigers
unzip -q lions-and-tigers.zip -d LionsTigers
ls LionsTigers
```

After unzipping, check the actual folder names Kaggle gives you (they may be nested, e.g. `LionsTigers/train/Lion` and `LionsTigers/train/Tiger`, or just `LionsTigers/Lion` and `LionsTigers/Tiger`). Update `data_dir` below to match — this notebook assumes a flat structure:

```
LionsTigers/
    Lion/
        *.jpg
    Tiger/
        *.jpg
```

If your dataset instead comes pre-split into `train/` and `test/` folders (like the iNaturalist Big Cats dataset), you can skip the `validation_split` step below and load `train/` and `test/` as two separate `image_dataset_from_directory` calls instead.

In [ ]:
import os
import numpy as np
import keras
from keras import layers
from tensorflow import data as tf_data
import matplotlib.pyplot as plt

## Filter out corrupted images

Same JFIF header check as the cats/dogs version — filters badly-encoded JPEGs before training.

In [ ]:
data_dir = "LionsTigers"

num_skipped = 0
for folder_name in ("Lion", "Tiger"):
    folder_path = os.path.join(data_dir, folder_name)
    for fname in os.listdir(folder_path):
        fpath = os.path.join(folder_path, fname)
        try:
            fobj = open(fpath, "rb")
            is_jfif = b"JFIF" in fobj.peek(10)
        finally:
            fobj.close()

        if not is_jfif:
            num_skipped += 1
            os.remove(fpath)

print(f"Deleted {num_skipped} images.")

## Generate a `Dataset`

Lion vs Tiger images are used at 180x180 like the original — bump this up (e.g. 224x224) if you want more detail and have the compute budget, since fur/stripe/mane texture matters more here than for cats vs dogs.

In [ ]:
image_size = (180, 180)
batch_size = 32  # smaller than the 128 used for cats/dogs — this dataset is much smaller (only ~400 images)

train_ds, val_ds = keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="both",
    seed=1337,
    image_size=image_size,
    batch_size=batch_size,
)

class_names = train_ds.class_names
print("Classes:", class_names)  # alphabetical -> ['Lion', 'Tiger'], so label 0 = Lion, 1 = Tiger

## Visualize the data

First 9 images in the training set, labeled 0 (Lion) or 1 (Tiger).

In [ ]:
plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(np.array(images[i]).astype("uint8"))
        plt.title(class_names[int(labels[i])])
        plt.axis("off")

## Using image data augmentation

Same idea as cats/dogs. For lions/tigers, a horizontal flip and small rotation are safe (their manes/stripes don't have a fixed left-right orientation), but avoid heavy color jitter — mane color vs. stripe pattern is part of what actually separates the two classes.

In [ ]:
data_augmentation_layers = [
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
]


def data_augmentation(images):
    for layer in data_augmentation_layers:
        images = layer(images)
    return images

In [ ]:
plt.figure(figsize=(10, 10))
for images, _ in train_ds.take(1):
    for i in range(9):
        augmented_images = data_augmentation(images)
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(np.array(augmented_images[0]).astype("uint8"))
        plt.axis("off")

## Configure the dataset for performance

Apply augmentation to the training set and prefetch both.

In [ ]:
train_ds = train_ds.map(
    lambda img, label: (data_augmentation(img), label),
    num_parallel_calls=tf_data.AUTOTUNE,
)
train_ds = train_ds.prefetch(tf_data.AUTOTUNE)
val_ds = val_ds.prefetch(tf_data.AUTOTUNE)

## Build a model

Identical mini-Xception architecture to the cats/dogs notebook — this network doesn't care what the two classes are, it's a general-purpose small image classifier. With only ~400 images total, expect to lean harder on augmentation and possibly fewer epochs before overfitting than the 23k-image cats/dogs run.

In [ ]:
def make_model(input_shape, num_classes):
    inputs = keras.Input(shape=input_shape)

    # Entry block
    x = layers.Rescaling(1.0 / 255)(inputs)
    x = layers.Conv2D(128, 3, strides=2, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    previous_block_activation = x  # Set aside residual

    for size in [256, 512, 728]:
        x = layers.Activation("relu")(x)
        x = layers.SeparableConv2D(size, 3, padding="same")(x)
        x = layers.BatchNormalization()(x)

        x = layers.Activation("relu")(x)
        x = layers.SeparableConv2D(size, 3, padding="same")(x)
        x = layers.BatchNormalization()(x)

        x = layers.MaxPooling2D(3, strides=2, padding="same")(x)

        residual = layers.Conv2D(size, 1, strides=2, padding="same")(
            previous_block_activation
        )
        x = layers.add([x, residual])
        previous_block_activation = x

    x = layers.SeparableConv2D(1024, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    x = layers.GlobalAveragePooling2D()(x)
    if num_classes == 2:
        units = 1
    else:
        units = num_classes

    x = layers.Dropout(0.25)(x)
    outputs = layers.Dense(units, activation=None)(x)
    return keras.Model(inputs, outputs)


model = make_model(input_shape=image_size + (3,), num_classes=2)
keras.utils.plot_model(model, show_shapes=True)

## Train the model

With a much smaller dataset than cats/dogs (~400 vs ~23k images), watch `val_acc` closely — this model can overfit fast. If validation accuracy plateaus or drops early, cut `epochs` down or add `EarlyStopping`. Consider also trying transfer learning (e.g. a frozen `keras.applications.Xception` or `MobileNetV2` base) if accuracy stalls — small datasets usually benefit a lot from a pretrained backbone.

In [ ]:
epochs = 25

callbacks = [
    keras.callbacks.ModelCheckpoint("save_at_{epoch}.keras"),
]
model.compile(
    optimizer=keras.optimizers.Adam(3e-4),
    loss=keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=[keras.metrics.BinaryAccuracy(name="acc")],
)
model.fit(
    train_ds,
    epochs=epochs,
    callbacks=callbacks,
    validation_data=val_ds,
)

## Run inference on new data

Swap in the path to any lion or tiger image from your dataset (or a new one you find) to test the trained model. Data augmentation and dropout are inactive at inference time.

In [ ]:
img_path = "LionsTigers/Lion/1.jpg"  # change to an actual image path in your dataset
img = keras.utils.load_img(img_path, target_size=image_size)
plt.imshow(img)

img_array = keras.utils.img_to_array(img)
img_array = keras.ops.expand_dims(img_array, 0)  # Create batch axis

predictions = model.predict(img_array)
score = float(keras.ops.sigmoid(predictions[0][0]))
print(f"This image is {100 * (1 - score):.2f}% Lion and {100 * score:.2f}% Tiger.")